In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../data/raw/day-ahead prices.csv")




In [51]:
df.head()

,_id,start_time_sweden,start_time_utc,bidding_zone,price,price_unit,soda_hashbyte,soda_identity
0,1,2020-11-03T14:00:00,2020-11-03T13:00:00,SE2,8.63,EUR-MWh,0cc12819c25b03ed5ba3a8378ef7a777,1
1,2,2022-11-29T12:00:00,2022-11-29T11:00:00,SE2,397.79,EUR-MWh,595269e7875381dbcb56aaaac1f73e35,2
2,3,2023-01-14T12:00:00,2023-01-14T11:00:00,DELU,48.91,EUR-MWh,655354f4e53c4b66f002c67226d0eb37,3
3,4,2021-11-27T23:00:00,2021-11-27T22:00:00,NO2,119.96,EUR-MWh,88cddb43505175cb7687d5dde6748361,4
4,5,2024-06-29T03:00:00,2024-06-29T01:00:00,PL,111.58,EUR-MWh,faae0846cef5054654c5475577791293,5


In [52]:
df.shape

(1196959, 8)

In [53]:
se3 = df[df["bidding_zone"] == "SE3"].copy()

In [54]:
se3.head()

,_id,start_time_sweden,start_time_utc,bidding_zone,price,price_unit,soda_hashbyte,soda_identity
14,15,2021-05-21T00:00:00,2021-05-20T22:00:00,SE3,20.51,EUR-MWh,9970c9f43c21e41535922e328e9ffd86,15
16,17,2023-01-31T23:00:00,2023-01-31T22:00:00,SE3,68.83,EUR-MWh,ff340634016c8ffcd5224552d15c0b6b,17
32,33,2021-10-21T21:00:00,2021-10-21T19:00:00,SE3,14.81,EUR-MWh,4ab57fd98442cf565a1488ad0520ef4f,33
65,66,2020-09-05T10:00:00,2020-09-05T08:00:00,SE3,31.48,EUR-MWh,2ddf1b88057cc0648ecafad76d935178,66
87,88,2024-01-06T09:00:00,2024-01-06T08:00:00,SE3,94.98,EUR-MWh,cf63d828ec0560937eef9c1c1d364e5e,88


In [55]:
se3["bidding_zone"].unique()

<StringArray>
['SE3']
Length: 1, dtype: str

In [56]:
se3["timestamp"] = pd.to_datetime(
    se3["start_time_utc"],
    utc= True
)

In [57]:
se3["timestamp"].min(), se3["timestamp"].max()

(Timestamp('2019-12-31 23:00:00+0000', tz='UTC'),
 Timestamp('2026-07-01 21:45:00+0000', tz='UTC'))

In [58]:
se3 = se3.sort_values("timestamp").reset_index(drop=True)

In [59]:
se3.head()

,_id,start_time_sweden,start_time_utc,bidding_zone,price,price_unit,soda_hashbyte,soda_identity,timestamp
0,124233,2020-01-01T00:00:00,2019-12-31T23:00:00,SE3,28.78,EUR-MWh,9b05db3ac3b6d51c8f14ef5b4c953ec2,124233,2019-12-31 23:00:00+00:00
1,517847,2020-01-01T01:00:00,2020-01-01T00:00:00,SE3,28.45,EUR-MWh,156d0a86a1487b57c63a69976faab85a,517847,2020-01-01 00:00:00+00:00
2,138937,2020-01-01T02:00:00,2020-01-01T01:00:00,SE3,27.90,EUR-MWh,5b67a4ab28f48394d44aa39450846821,138937,2020-01-01 01:00:00+00:00
3,1061525,2020-01-01T03:00:00,2020-01-01T02:00:00,SE3,27.52,EUR-MWh,54be00269783110339d08d097a918bde,1061525,2020-01-01 02:00:00+00:00
4,623791,2020-01-01T04:00:00,2020-01-01T03:00:00,SE3,27.54,EUR-MWh,9286c3454a886c0e0b75429436bde0cc,623791,2020-01-01 03:00:00+00:00


In [60]:
start_date = pd.Timestamp("2024-03-08 00:00", tz="UTC")
end_date = pd.Timestamp("2026-05-01 00:00", tz="UTC")

se3_project = se3[
    (se3["timestamp"] >= start_date) & 
    (se3["timestamp"] < end_date)
].copy()

In [61]:
se3_project["timestamp"].min(), se3_project["timestamp"].max()

(Timestamp('2024-03-08 00:00:00+0000', tz='UTC'),
 Timestamp('2026-04-30 23:45:00+0000', tz='UTC'))

In [68]:
hourly_counts = (
    se3_project
    .set_index("timestamp")
    ["price"]
    .resample("1h")
    .count()
)

In [69]:
hourly_counts.value_counts().sort_index()

price
0      297
1    13629
2       25
3      130
4     4735
Name: count, dtype: int64

In [70]:
transition = pd.Timestamp("2025-09-30 22:00", tz="UTC")

In [71]:
hourly_check = hourly_counts.reset_index()
hourly_check.columns = ["timestamp", "count"]

In [72]:
hourly_check["expected"] = 1

hourly_check.loc[
    hourly_check["timestamp"] >= transition,
    "expected"
] = 4

In [73]:
issues = hourly_check[
    hourly_check["count"] != hourly_check["expected"]
].copy()

In [74]:
issues.head(20)

,timestamp,count,expected
31,2024-03-09 07:00:00+00:00,0,1
168,2024-03-15 00:00:00+00:00,0,1
579,2024-04-01 03:00:00+00:00,0,1
721,2024-04-07 01:00:00+00:00,0,1
722,2024-04-07 02:00:00+00:00,0,1
723,2024-04-07 03:00:00+00:00,0,1
724,2024-04-07 04:00:00+00:00,0,1
725,2024-04-07 05:00:00+00:00,0,1
726,2024-04-07 06:00:00+00:00,0,1
815,2024-04-10 23:00:00+00:00,0,1


In [75]:
issues.tail(20)

,timestamp,count,expected
18238,2026-04-06 22:00:00+00:00,3,4
18240,2026-04-07 00:00:00+00:00,3,4
18336,2026-04-11 00:00:00+00:00,3,4
18338,2026-04-11 02:00:00+00:00,3,4
18413,2026-04-14 05:00:00+00:00,3,4
18610,2026-04-22 10:00:00+00:00,3,4
18611,2026-04-22 11:00:00+00:00,3,4
18625,2026-04-23 01:00:00+00:00,3,4
18636,2026-04-23 12:00:00+00:00,3,4
18660,2026-04-24 12:00:00+00:00,2,4


In [76]:
issues = issues.sort_values("timestamp").reset_index(drop=True)

issues["new_block"] = (
    issues["timestamp"].diff() != pd.Timedelta(hours=1)
)

issues["block"] = issues["new_block"].cumsum()

issue_blocks = issues.groupby("block").agg(
    start=("timestamp", "min"),
    end=("timestamp", "max"),
    hours=("timestamp", "size")
)

issue_blocks

,start,end,hours
block,,,
1,2024-03-09 07:00:00+00:00,2024-03-09 07:00:00+00:00,1
2,2024-03-15 00:00:00+00:00,2024-03-15 00:00:00+00:00,1
3,2024-04-01 03:00:00+00:00,2024-04-01 03:00:00+00:00,1
4,2024-04-07 01:00:00+00:00,2024-04-07 06:00:00+00:00,6
5,2024-04-10 23:00:00+00:00,2024-04-10 23:00:00+00:00,1
...,...,...,...
216,2026-04-26 09:00:00+00:00,2026-04-26 09:00:00+00:00,1
217,2026-04-26 11:00:00+00:00,2026-04-26 13:00:00+00:00,3
218,2026-04-27 11:00:00+00:00,2026-04-27 11:00:00+00:00,1


In [77]:
issue_blocks.sort_values("hours", ascending=False).head(20)

,start,end,hours
block,,,
84,2025-09-30 22:00:00+00:00,2025-10-08 21:00:00+00:00,192
4,2024-04-07 01:00:00+00:00,2024-04-07 06:00:00+00:00,6
15,2024-06-30 09:00:00+00:00,2024-06-30 13:00:00+00:00,5
31,2024-08-11 00:00:00+00:00,2024-08-11 04:00:00+00:00,5
199,2026-03-25 01:00:00+00:00,2026-03-25 03:00:00+00:00,3
157,2025-12-27 23:00:00+00:00,2025-12-28 01:00:00+00:00,3
40,2024-09-01 01:00:00+00:00,2024-09-01 03:00:00+00:00,3
144,2025-12-19 22:00:00+00:00,2025-12-20 00:00:00+00:00,3
217,2026-04-26 11:00:00+00:00,2026-04-26 13:00:00+00:00,3


In [78]:
hourly_price = (
    se3_project
    .set_index("timestamp")["price"]
    .resample("1h")
    .mean()
)

hourly_count = (
    se3_project
    .set_index("timestamp")["price"]
    .resample("1h")
    .count()
)

price_hourly = pd.DataFrame({
    "price": hourly_price,
    "count": hourly_count
}).reset_index()

In [79]:
transition = pd.Timestamp("2025-09-30 22:00", tz="UTC")

In [80]:
price_hourly["valid"] = (
    (
        (price_hourly["timestamp"] < transition) &
        (price_hourly["count"] == 1)
    )
    |
    (
        (price_hourly["timestamp"] >= transition) &
        (price_hourly["count"] == 4)
    )
)

In [81]:
price_hourly.loc[
    ~price_hourly["valid"],
    "price"
] = pd.NA

In [82]:
price_hourly["valid"].value_counts()

valid
True     18357
False      459
Name: count, dtype: int64

In [83]:
price_hourly["price"].isna().sum()

np.int64(459)

In [84]:
train_prices = price_hourly[
    (price_hourly["timestamp"] >= "2024-03-08") &
    (price_hourly["timestamp"] < "2025-03-08")
]

validation_prices = price_hourly[
    (price_hourly["timestamp"] >= "2025-03-08") &
    (price_hourly["timestamp"] < "2025-05-01")
]

test_prices = price_hourly[
    (price_hourly["timestamp"] >= "2025-05-01") &
    (price_hourly["timestamp"] < "2026-05-01")
]

print("Train missing:", train_prices["price"].isna().sum())
print("Validation missing:", validation_prices["price"].isna().sum())
print("Test missing:", test_prices["price"].isna().sum())

Train missing: 92
Validation missing: 3
Test missing: 364


In [85]:
price_hourly.to_csv(
    "../data/processed/se3_hourly_prices.csv",
    index=False
)